In [27]:
import os
import json
from openai import OpenAI
from google.colab import userdata

import numpy as np
from sklearn.metrics.pairwise import cosine_similarity
from uuid import uuid4 as uuid

In [3]:
client = OpenAI(api_key=userdata.get('openai_IK'))

In [4]:
mock_document = [
  {
    "ticket_id": "TICK-001",
    "title": "Users unable to log in after password reset",
    "description": "Multiple users reporting authentication failures after performing password reset. Error message: 'Invalid credentials'. Issue started after recent security patch deployment.",
    "resolution": "Found that the password hash algorithm was updated but session tokens weren't invalidated. Solution: Clear all active sessions and force re-authentication. Implemented automatic session cleanup on password change.",
    "category": "Authentication",
    "priority": "High",
    "created_date": "2024-01-15",
    "resolved_date": "2024-01-15"
  },
  {
    "ticket_id": "TICK-002",
    "title": "Database connection timeout in production",
    "description": "Application experiencing intermittent 500 errors. Logs show 'connection pool exhausted' and 'timeout waiting for connection'. Affects approximately 20% of requests during peak hours.",
    "resolution": "Database connection pool was sized too small for peak load. Increased max_connections from 20 to 100. Added connection pooling monitoring and alerts. Optimized long-running queries that were holding connections.",
    "category": "Database",
    "priority": "Critical",
    "created_date": "2024-01-18",
    "resolved_date": "2024-01-18"
  },
  {
    "ticket_id": "TICK-003",
    "title": "Payment processing fails for international cards",
    "description": "Customers using non-US credit cards receiving 'Payment declined' errors. Payment gateway returns error code 402. Issue only affects cards issued outside United States.",
    "resolution": "Payment gateway configuration was missing international card processing flag. Updated gateway settings to enable international transactions. Added country-specific currency conversion. Retested with cards from EU, UK, and APAC regions.",
    "category": "Payment",
    "priority": "High",
    "created_date": "2024-01-20",
    "resolved_date": "2024-01-21"
  },
  {
    "ticket_id": "TICK-004",
    "title": "Mobile app crashes on iOS 17",
    "description": "App crashes immediately on launch for users who upgraded to iOS 17. Crash reports show EXC_BAD_ACCESS in networking module. Affects approximately 30% of iOS user base.",
    "resolution": "iOS 17 deprecated certain networking APIs we were using. Updated URLSession implementation to use new async/await pattern. Removed deprecated NSURLConnection usage. Submitted updated build to App Store with iOS 17 compatibility.",
    "category": "Mobile",
    "priority": "Critical",
    "created_date": "2024-01-22",
    "resolved_date": "2024-01-23"
  },
  {
    "ticket_id": "TICK-005",
    "title": "Memory leak in background worker process",
    "description": "Background job processor showing steady memory increase over time. Process killed by OS after reaching 8GB memory usage. Requires daily restart to maintain functionality.",
    "resolution": "Memory profiling revealed unclosed database cursors in batch processing loop. Added explicit cursor.close() calls and implemented context managers for all DB operations. Memory usage now stable at 500MB. Set up memory usage monitoring alerts.",
    "category": "Performance",
    "priority": "High",
    "created_date": "2024-01-25",
    "resolved_date": "2024-01-26"
  },
  {
    "ticket_id": "TICK-006",
    "title": "Email notifications not being delivered",
    "description": "Users reporting they're not receiving password reset and confirmation emails. SMTP server logs show successful sends, but emails never arrive in inbox or spam folder.",
    "resolution": "Email service provider had blacklisted our sending IP due to spam complaints. Implemented SPF, DKIM, and DMARC records. Switched to dedicated IP address with proper warmup process. Added email deliverability monitoring.",
    "category": "Email",
    "priority": "High",
    "created_date": "2024-01-28",
    "resolved_date": "2024-01-29"
  },
  {
    "ticket_id": "TICK-007",
    "title": "Search functionality returns no results",
    "description": "Product search returning empty results for queries that should match. Elasticsearch cluster health showing yellow status. Index size growing abnormally large.",
    "resolution": "Elasticsearch index had corrupted shards after power outage. Performed cluster health check, deleted corrupted index, and rebuilt from database. Implemented automated daily index health checks and backup strategy.",
    "category": "Search",
    "priority": "Critical",
    "created_date": "2024-02-01",
    "resolved_date": "2024-02-01"
  },
  {
    "ticket_id": "TICK-008",
    "title": "File uploads failing for large files",
    "description": "Users unable to upload files larger than 5MB. Progress bar reaches 100% but then shows 'Upload failed' error. Works fine for smaller files. Browser console shows 413 error.",
    "resolution": "Nginx was configured with default 1MB request body size limit. Updated nginx.conf to set client_max_body_size to 50MB. Also increased Node.js Express body-parser limit. Added validation to show friendly error for files exceeding 50MB.",
    "category": "File Upload",
    "priority": "Medium",
    "created_date": "2024-02-03",
    "resolved_date": "2024-02-03"
  },
  {
    "ticket_id": "TICK-009",
    "title": "API rate limiting too aggressive",
    "description": "Legitimate users hitting rate limits during normal usage. Current limit is 100 requests per minute. Users getting 429 Too Many Requests errors, especially when using batch operations.",
    "resolution": "Analyzed API usage patterns and found limit was indeed too restrictive. Increased rate limit to 500 requests per minute for authenticated users. Implemented tiered rate limiting based on account type. Added X-RateLimit headers to API responses.",
    "category": "API",
    "priority": "Medium",
    "created_date": "2024-02-05",
    "resolved_date": "2024-02-06"
  },
  {
    "ticket_id": "TICK-010",
    "title": "Dashboard loading extremely slowly",
    "description": "Admin dashboard takes 30+ seconds to load. Browser DevTools show multiple slow database queries. Page becomes unresponsive during load. Affects all users with admin privileges.",
    "resolution": "Dashboard was making N+1 queries for user statistics. Implemented query optimization using JOIN statements and added database indexes on frequently queried columns. Introduced Redis caching for aggregate statistics with 5-minute TTL. Load time reduced to under 2 seconds.",
    "category": "Performance",
    "priority": "High",
    "created_date": "2024-02-08",
    "resolved_date": "2024-02-09"
  },
  {
    "ticket_id": "TICK-011",
    "title": "SSO authentication broken after upgrade",
    "description": "Single Sign-On integration with corporate SAML provider failing. Users redirected to SSO portal but then receive 'Invalid SAML response' error when redirected back. Started after v2.5 upgrade.",
    "resolution": "Upgrade changed default SAML signature algorithm from SHA-1 to SHA-256, but IdP configuration wasn't updated. Coordinated with IT team to update IdP settings. Added configuration option to specify signature algorithm. Tested with multiple SAML providers.",
    "category": "Authentication",
    "priority": "Critical",
    "created_date": "2024-02-10",
    "resolved_date": "2024-02-11"
  },
  {
    "ticket_id": "TICK-012",
    "title": "Scheduled reports not generating",
    "description": "Daily scheduled reports stopped generating three days ago. Cron job shows as running in logs but no PDF files are created. Manual report generation works correctly.",
    "resolution": "Disk space was full on the reports server (100% usage). Old report files were not being cleaned up. Implemented automated cleanup of reports older than 30 days. Increased disk space and added disk usage monitoring alerts.",
    "category": "Reporting",
    "priority": "Medium",
    "created_date": "2024-02-12",
    "resolved_date": "2024-02-12"
  },
  {
    "ticket_id": "TICK-013",
    "title": "Webhook deliveries failing intermittently",
    "description": "Outgoing webhooks to customer endpoints timing out. Retry logic exhausting all attempts. Some webhooks succeed while others fail for same endpoint. No pattern to failures.",
    "resolution": "Default timeout was 5 seconds but some customer endpoints were slow. Increased timeout to 30 seconds. Implemented exponential backoff for retries. Added webhook delivery status dashboard for customers to monitor. Persisted failed webhooks to dead letter queue.",
    "category": "Integration",
    "priority": "High",
    "created_date": "2024-02-14",
    "resolved_date": "2024-02-15"
  },
  {
    "ticket_id": "TICK-014",
    "title": "Session timeout too short",
    "description": "Users complaining about being logged out too frequently. Currently set to 15 minutes. Particularly problematic for users filling out long forms, losing unsaved data when session expires.",
    "resolution": "Extended session timeout to 2 hours for active users. Implemented activity-based session renewal (extends on any user action). Added auto-save functionality for forms with localStorage backup. Implemented 'session about to expire' warning modal.",
    "category": "Authentication",
    "priority": "Medium",
    "created_date": "2024-02-16",
    "resolved_date": "2024-02-17"
  },
  {
    "ticket_id": "TICK-015",
    "title": "Export to CSV producing corrupt files",
    "description": "Data export feature generating CSV files that can't be opened in Excel. File appears to have encoding issues with special characters. Particularly affects records with non-ASCII characters.",
    "resolution": "Export was using default UTF-8 encoding without BOM, causing Excel to misinterpret character encoding. Changed to UTF-8 with BOM for Excel compatibility. Added proper escaping for special characters in CSV fields. Included character encoding in filename.",
    "category": "Export",
    "priority": "Medium",
    "created_date": "2024-02-18",
    "resolved_date": "2024-02-19"
  },
  {
    "ticket_id": "TICK-016",
    "title": "Two-factor authentication codes not working",
    "description": "Users unable to log in with TOTP codes from authenticator apps. Codes shown as invalid even when entered immediately after generation. Affects random subset of users.",
    "resolution": "Server time was drifting by 2 minutes due to misconfigured NTP. TOTP codes are time-sensitive and were outside acceptance window. Configured NTP synchronization and verified time accuracy. Added time drift monitoring. Increased TOTP acceptance window to ±1 time step.",
    "category": "Authentication",
    "priority": "High",
    "created_date": "2024-02-20",
    "resolved_date": "2024-02-20"
  },
  {
    "ticket_id": "TICK-017",
    "title": "Real-time notifications delayed by several minutes",
    "description": "WebSocket notifications arriving 3-5 minutes late. Users seeing chat messages and alerts significantly delayed. WebSocket connection appears stable, no disconnections in logs.",
    "resolution": "Message broker (RabbitMQ) had accumulated a large queue backlog. Consumer was processing messages slower than they were being produced. Increased number of consumer workers from 2 to 10. Optimized message processing logic. Added queue length monitoring.",
    "category": "Real-time",
    "priority": "High",
    "created_date": "2024-02-22",
    "resolved_date": "2024-02-23"
  },
  {
    "ticket_id": "TICK-018",
    "title": "API returning inconsistent data",
    "description": "Same API endpoint returning different results for identical requests within seconds. Appears to be caching issue. GET /api/users/:id sometimes returns stale data.",
    "resolution": "Load balancer was routing to 3 app servers with separate local caches. Cache invalidation only happening on the server that processed the update. Migrated to centralized Redis cache shared across all servers. Implemented cache invalidation via pub/sub.",
    "category": "API",
    "priority": "Critical",
    "created_date": "2024-02-25",
    "resolved_date": "2024-02-26"
  },
  {
    "ticket_id": "TICK-019",
    "title": "Image thumbnails not generating",
    "description": "Newly uploaded images not showing thumbnails. Background job for thumbnail generation appears to be stuck. Queue shows 5000+ pending jobs. Old thumbnails still display correctly.",
    "resolution": "ImageMagick process was hanging on certain corrupted image files. Implemented timeout for image processing operations. Added validation to reject corrupted files at upload. Cleared stuck jobs and reprocessed with timeout protection. Added failure logging for investigation.",
    "category": "Media Processing",
    "priority": "Medium",
    "created_date": "2024-02-27",
    "resolved_date": "2024-02-28"
  },
  {
    "ticket_id": "TICK-020",
    "title": "Cannot delete user accounts",
    "description": "Delete account functionality failing with foreign key constraint errors. Database shows referential integrity violations. Affects GDPR compliance for user data deletion requests.",
    "resolution": "Deletion logic was missing cascading deletes for related records. Implemented proper cleanup order: delete user sessions, then activities, then preferences, then user record. Added soft delete option for audit trail. Created GDPR-compliant data anonymization as alternative to hard delete.",
    "category": "User Management",
    "priority": "High",
    "created_date": "2024-03-01",
    "resolved_date": "2024-03-02"
  }
]

In [5]:
mock_document[0]

{'ticket_id': 'TICK-001',
 'title': 'Users unable to log in after password reset',
 'description': "Multiple users reporting authentication failures after performing password reset. Error message: 'Invalid credentials'. Issue started after recent security patch deployment.",
 'resolution': "Found that the password hash algorithm was updated but session tokens weren't invalidated. Solution: Clear all active sessions and force re-authentication. Implemented automatic session cleanup on password change.",
 'category': 'Authentication',
 'priority': 'High',
 'created_date': '2024-01-15',
 'resolved_date': '2024-01-15'}

## Process Data/Document

In [6]:
from langchain_core.documents import Document

processed_documents = []

for document in mock_document:
  processed_document = f"""
    Ticket ID: {document['ticket_id']}
    Title: {document['title']}
    Description: {document['description']}
    Resolution: {document['resolution']}
  """.strip()

  processed_documents.append(Document(page_content=processed_document))

print(processed_documents[0])

print(f"\nTotal documents: {len(processed_documents)}")
print(f"Avg document length: {sum(len(d.page_content) for d in processed_documents) // len(processed_documents)} chars")

page_content='Ticket ID: TICK-001
    Title: Users unable to log in after password reset
    Description: Multiple users reporting authentication failures after performing password reset. Error message: 'Invalid credentials'. Issue started after recent security patch deployment.
    Resolution: Found that the password hash algorithm was updated but session tokens weren't invalidated. Solution: Clear all active sessions and force re-authentication. Implemented automatic session cleanup on password change.'

Total documents: 20
Avg document length: 520 chars


## Fixed Character Split

In [ ]:
# pip install -U langchain-text-splitters

In [10]:
from langchain_text_splitters import CharacterTextSplitter, RecursiveCharacterTextSplitter

In [11]:
splitter = CharacterTextSplitter(
    separator = " ",
    chunk_size = 200,
    chunk_overlap  = 50,
    length_function = len
)

chunks = splitter.split_documents(processed_documents)

print(chunks[0].page_content)
print("*"*100)
print(chunks[1].page_content)
print("*"*100)
print(chunks[3].page_content)

Ticket ID: TICK-001
 Title: Users unable to log in after password reset
 Description: Multiple users reporting authentication failures after performing password reset. Error message: 'Invalid
****************************************************************************************************
performing password reset. Error message: 'Invalid credentials'. Issue started after recent security patch deployment.
 Resolution: Found that the password hash algorithm was updated but session
****************************************************************************************************
Ticket ID: TICK-002
 Title: Database connection timeout in production
 Description: Application experiencing intermittent 500 errors. Logs show 'connection pool exhausted' and 'timeout waiting for


In [12]:
splitter = CharacterTextSplitter(
    separator = "\n",
    chunk_size = 300,
    chunk_overlap  = 50,
    length_function = len
)

chunks = splitter.split_documents(processed_documents)

print(chunks[0].page_content)
print("*"*100)
print(chunks[1].page_content)

Ticket ID: TICK-001
    Title: Users unable to log in after password reset
    Description: Multiple users reporting authentication failures after performing password reset. Error message: 'Invalid credentials'. Issue started after recent security patch deployment.
****************************************************************************************************
Resolution: Found that the password hash algorithm was updated but session tokens weren't invalidated. Solution: Clear all active sessions and force re-authentication. Implemented automatic session cleanup on password change.


In [13]:
# 1. Join all ticket texts with your double newline separator into one giant string
giant_string = "\n\n".join([d.page_content for d in processed_documents])

# 2. Use a smaller chunk size to force splits across the boundaries
splitter = CharacterTextSplitter(
    separator = "\n\n",
    chunk_size = 600,
    chunk_overlap = 100,
    length_function = len
)

chunks = splitter.split_text(giant_string)

print(chunks[0])
print("*"*100)
print(chunks[1])

Ticket ID: TICK-001
    Title: Users unable to log in after password reset
    Description: Multiple users reporting authentication failures after performing password reset. Error message: 'Invalid credentials'. Issue started after recent security patch deployment.
    Resolution: Found that the password hash algorithm was updated but session tokens weren't invalidated. Solution: Clear all active sessions and force re-authentication. Implemented automatic session cleanup on password change.
****************************************************************************************************
Ticket ID: TICK-002
    Title: Database connection timeout in production
    Description: Application experiencing intermittent 500 errors. Logs show 'connection pool exhausted' and 'timeout waiting for connection'. Affects approximately 20% of requests during peak hours.
    Resolution: Database connection pool was sized too small for peak load. Increased max_connections from 20 to 100. Added connec

### Chunking + Embedding + Query

In [14]:
splitter = CharacterTextSplitter(
    separator = "\n\n",
    chunk_size = 600,
    chunk_overlap  = 50,
    length_function = len
)

chunks = splitter.split_documents(processed_documents)

print(chunks[0].page_content)
print("*"*100)
print(chunks[1].page_content)

Ticket ID: TICK-001
    Title: Users unable to log in after password reset
    Description: Multiple users reporting authentication failures after performing password reset. Error message: 'Invalid credentials'. Issue started after recent security patch deployment.
    Resolution: Found that the password hash algorithm was updated but session tokens weren't invalidated. Solution: Clear all active sessions and force re-authentication. Implemented automatic session cleanup on password change.
****************************************************************************************************
Ticket ID: TICK-002
    Title: Database connection timeout in production
    Description: Application experiencing intermittent 500 errors. Logs show 'connection pool exhausted' and 'timeout waiting for connection'. Affects approximately 20% of requests during peak hours.
    Resolution: Database connection pool was sized too small for peak load. Increased max_connections from 20 to 100. Added connec

In [15]:
chunk_list = [chunk.page_content for chunk in chunks]

response = client.embeddings.create(model="text-embedding-3-small", input=chunk_list)

embeddings = [record.embedding for record in response.data]

In [16]:
user_query = "Database is timing out frequently"

user_embedding = client.embeddings.create(model="text-embedding-3-small", input=user_query).data[0].embedding

score = cosine_similarity(np.array(user_embedding).reshape(1,-1), np.array(embeddings).reshape(len(embeddings),-1))[0]

print(f"Similar Chunk with score {score[score.argmax()]} is \n {chunk_list[score.argmax()]}")

Similar Chunk with score 0.558217989129302 is 
 Ticket ID: TICK-002
    Title: Database connection timeout in production
    Description: Application experiencing intermittent 500 errors. Logs show 'connection pool exhausted' and 'timeout waiting for connection'. Affects approximately 20% of requests during peak hours.
    Resolution: Database connection pool was sized too small for peak load. Increased max_connections from 20 to 100. Added connection pooling monitoring and alerts. Optimized long-running queries that were holding connections.


## Recursive Chunking

In [17]:
splitter = RecursiveCharacterTextSplitter(
    separators = ["\n\n", "\n", " "],
    chunk_size = 450,
    chunk_overlap  = 20,
    length_function = len
)

chunks = splitter.split_documents(processed_documents)

chunk_list = [chunk.page_content for chunk in chunks]

response = client.embeddings.create(model="text-embedding-3-small", input=chunk_list)

embeddings = [record.embedding for record in response.data]


In [18]:
user_query = "Database is timing out frequently"

user_embedding = client.embeddings.create(model="text-embedding-3-small", input=user_query).data[0].embedding

score = cosine_similarity(np.array(user_embedding).reshape(1,-1), np.array(embeddings).reshape(len(embeddings),-1))[0]

top_3_indices = score.argsort()[-3:][::-1]

joined_context = "\n\n".join([chunk_list[i] for i in top_3_indices if score[i] > 0.5])

print(joined_context)

Ticket ID: TICK-002
    Title: Database connection timeout in production
    Description: Application experiencing intermittent 500 errors. Logs show 'connection pool exhausted' and 'timeout waiting for connection'. Affects approximately 20% of requests during peak hours.


In [19]:
for i in score.argsort()[-3:][::-1]:
  print(score[i])

0.5641097070115209
0.4811578147920659
0.4618686497867789


### Recursive with metadata filtering

In [20]:
def json_table(embeddings, chunks):
  table_data = []

  for idx, (chunk, embedding) in enumerate(zip(chunks, embeddings)):

    row = {
        "chunk_id": f"chunk_{idx:03d}",
        "chunk_text": chunk.page_content,
        "chunk_embedding": embedding,
        "metadata": chunk.metadata
    }

    table_data.append(row)
  print(f"Processed json table")

  return table_data

In [21]:
splitter = RecursiveCharacterTextSplitter(
    separators = ["\n\n", "\n", " "],
    chunk_size = 500,
    chunk_overlap  = 20,
    length_function = len
)


processed_documents = []

for document in mock_document:
    # 1. Define the page content just like you did
    processed_document = f"""
      Ticket ID: {document['ticket_id']}
      Title: {document['title']}
      Description: {document['description']}
      Resolution: {document['resolution']}
    """.strip()

    # 2. Extract the fields you want to use for metadata filtering
    metadata = {
        "ticket_id": document["ticket_id"],
        "category": document["category"],
        "priority": document["priority"],
        "created_date": document["created_date"],
        "resolution": document["resolution"]
    }

    # 3. Pass both page_content and metadata to the Document object
    processed_documents.append(
        Document(page_content=processed_document, metadata=metadata)
    )

chunks = splitter.split_documents(processed_documents)

chunk_list = [chunk.page_content for chunk in chunks]

response = client.embeddings.create(model="text-embedding-3-small", input=chunk_list)

embeddings = [record.embedding for record in response.data]

jsonTable = json_table(embeddings, chunks)


Processed json table


In [22]:
for k,v in jsonTable[0].items():
  print(k,v)

chunk_id chunk_000
chunk_text Ticket ID: TICK-001
      Title: Users unable to log in after password reset
      Description: Multiple users reporting authentication failures after performing password reset. Error message: 'Invalid credentials'. Issue started after recent security patch deployment.
chunk_embedding [-0.0072479248046875, 0.024261474609375, 0.0238494873046875, 0.0027828216552734375, -0.019134521484375, -0.03277587890625, -0.0114898681640625, 0.097412109375, -0.0167083740234375, -0.006011962890625, 0.027801513671875, -7.212162017822266e-05, -0.0247039794921875, 0.007274627685546875, 0.01263427734375, 0.0208740234375, 0.0250701904296875, 0.02215576171875, -0.078369140625, 0.048370361328125, 0.032257080078125, 0.02716064453125, 0.005550384521484375, -0.003360748291015625, 0.005733489990234375, 0.041748046875, -0.01519012451171875, -0.03662109375, 0.05224609375, 0.004482269287109375, 0.01406097412109375, -0.027435302734375, -0.01092529296875, 0.048187255859375, 0.015281677246

In [23]:
user_query = "Database is timing out frequently"

user_embedding = client.embeddings.create(model="text-embedding-3-small", input=user_query).data[0].embedding

scores = []

for i, row in enumerate(jsonTable):
  if row["metadata"]["category"] == "Authentication":
    scores.append(cosine_similarity(np.array(user_embedding).reshape(1,-1), np.array(row["chunk_embedding"]).reshape(1,-1))[0][0])

scores = np.array(scores)
final_score = np.array(scores).reshape(1,-1)[0]

top_3_indices = final_score.argsort()[-3:][::-1]

joined_text = "\n\n".join([jsonTable[i]["chunk_text"] for i in top_3_indices])

print(joined_text)

Ticket ID: TICK-003
      Title: Payment processing fails for international cards
      Description: Customers using non-US credit cards receiving 'Payment declined' errors. Payment gateway returns error code 402. Issue only affects cards issued outside United States.

Ticket ID: TICK-005
      Title: Memory leak in background worker process
      Description: Background job processor showing steady memory increase over time. Process killed by OS after reaching 8GB memory usage. Requires daily restart to maintain functionality.

Resolution: Payment gateway configuration was missing international card processing flag. Updated gateway settings to enable international transactions. Added country-specific currency conversion. Retested with cards from EU, UK, and APAC regions.


In [24]:
user_query = "Database is timing out frequently"

user_embedding = client.embeddings.create(model="text-embedding-3-small", input=user_query).data[0].embedding

all_embeddings = np.array([row["chunk_embedding"] for row in jsonTable]).reshape(-1,1536)

scores = cosine_similarity(np.array(user_embedding).reshape(1,-1), all_embeddings)[0]

scores = np.array(scores)
final_score = np.array(scores).reshape(1,-1)[0]

top_3_indices = final_score.argsort()[-3:][::-1]

resolutions = [jsonTable[i]["metadata"]["resolution"] for i in top_3_indices]

joined_text = "\n\n".join(list(dict.fromkeys(resolutions)))

print(joined_text)

Database connection pool was sized too small for peak load. Increased max_connections from 20 to 100. Added connection pooling monitoring and alerts. Optimized long-running queries that were holding connections.

Extended session timeout to 2 hours for active users. Implemented activity-based session renewal (extends on any user action). Added auto-save functionality for forms with localStorage backup. Implemented 'session about to expire' warning modal.


In [25]:
for i in final_score.argsort()[-3:][::-1]:
  print(final_score[i])

0.5607507823964728
0.4811578147920659
0.46452560637278717


## Parent Child Chunking

In [37]:
parent_splitter = RecursiveCharacterTextSplitter(
    separators = ["\n\n", "\n", " "],
    chunk_size = 520,
    chunk_overlap  = 20,
    length_function = len
)

child_splitter = RecursiveCharacterTextSplitter(
    separators = ["\n\n", "\n", " "],
    chunk_size = 200,
    chunk_overlap  = 10,
    length_function = len
)

In [38]:
parent_child_table = []

for doc in processed_documents:
  parents = parent_splitter.split_documents([doc])

  for parent_chunk in parents:
    parent_id = str(uuid())

    children = child_splitter.split_documents([parent_chunk])

    for child_chunk in children:
      child_metadata = parent_chunk.metadata.copy()
      child_metadata['parent_id'] = parent_id
      child_metadata['parent_text'] = parent_chunk.page_content
      child_metadata['global_metadata'] = doc.metadata

      parent_child_table.append({
          "child_id": str(uuid()),
          "parent_id": parent_id,
          "child_text": child_chunk.page_content,
          "metadata": child_metadata,

      })

In [39]:
chunks = [row["child_text"] for row in parent_child_table]

response = client.embeddings.create(model="text-embedding-3-small", input=chunks)

for idx, record in enumerate(response.data):
    parent_child_table[idx]["child_embedding"] = record.embedding

print("Child embeddings generated and mapped.")

Child embeddings generated and mapped.


In [52]:
user_query = "Database is timing out frequently"

user_embedding = client.embeddings.create(model="text-embedding-3-small", input=user_query).data[0].embedding

child_matrix = np.array([row["child_embedding"] for row in parent_child_table]).reshape(-1, 1536)
scores = cosine_similarity(np.array(user_embedding).reshape(1, -1), child_matrix)[0]

top_indices = scores.argsort()[-5:][::-1]

retrieved_parents = [parent_child_table[i]["metadata"]["parent_text"] for i in top_indices]

unique_parents = list(dict.fromkeys(retrieved_parents))

print("\n\n".join([parent for parent in unique_parents]))

Ticket ID: TICK-002
      Title: Database connection timeout in production
      Description: Application experiencing intermittent 500 errors. Logs show 'connection pool exhausted' and 'timeout waiting for connection'. Affects approximately 20% of requests during peak hours.
      Resolution: Database connection pool was sized too small for peak load. Increased max_connections from 20 to 100. Added connection pooling monitoring and alerts. Optimized long-running queries that were holding connections.

Ticket ID: TICK-014
      Title: Session timeout too short
      Description: Users complaining about being logged out too frequently. Currently set to 15 minutes. Particularly problematic for users filling out long forms, losing unsaved data when session expires.

Ticket ID: TICK-013
      Title: Webhook deliveries failing intermittently
      Description: Outgoing webhooks to customer endpoints timing out. Retry logic exhausting all attempts. Some webhooks succeed while others fail for

In [62]:
user_query = "Database is timing out frequently"

user_embedding = client.embeddings.create(model="text-embedding-3-small", input=user_query).data[0].embedding

child_matrix = np.array([row["child_embedding"] for row in parent_child_table]).reshape(-1, 1536)
scores = cosine_similarity(np.array(user_embedding).reshape(1, -1), child_matrix)[0]

top_indices = scores.argsort()[-5:][::-1]

retrieved_global_metadata = [parent_child_table[i]["metadata"]["global_metadata"] for i in top_indices if scores[i] > 0.5]

In [63]:
resolutions = []

for doc in retrieved_global_resolution:
  for k, v in doc.items():
    if k == "resolution":
      resolutions.append(v)

print("\n\n".join(list(dict.fromkeys(resolutions))))

Database connection pool was sized too small for peak load. Increased max_connections from 20 to 100. Added connection pooling monitoring and alerts. Optimized long-running queries that were holding connections.

Extended session timeout to 2 hours for active users. Implemented activity-based session renewal (extends on any user action). Added auto-save functionality for forms with localStorage backup. Implemented 'session about to expire' warning modal.

Default timeout was 5 seconds but some customer endpoints were slow. Increased timeout to 30 seconds. Implemented exponential backoff for retries. Added webhook delivery status dashboard for customers to monitor. Persisted failed webhooks to dead letter queue.
